# M41 lab solution: an agent as a service

**Brief.** Agent as a service: put your agentic RAG graph behind A2A and call it from an agent built in a different framework.

- **Server side.** A small agentic RAG graph in LangGraph (retrieve, grade, rewrite once if the passages do not answer, generate), rebuilt inline over the Larkspur handbook. An `AgentExecutor` wraps it and `A2AStarletteApplication` serves it with an Agent Card, as in notebook `02a_langgraph_agent_over_a2a.ipynb`. Uvicorn runs it in a background thread.
- **Client side.** A support agent written with the OpenAI Python SDK and no agent framework, like the plain-LLM agent of notebook `03a_cross_framework_agent_interop.ipynb`. It discovers the server through its Agent Card and has one tool, `ask_handbook_agent`, which sends an A2A message and returns the reply.

The two sides share no code and no framework: only the A2A protocol over HTTP.

## Setup

In [ ]:
import asyncio
import json
import os
import sys
import threading
import time
from pathlib import Path
from typing import Literal
from uuid import uuid4

import httpx
import nest_asyncio
import uvicorn
from a2a.client import A2ACardResolver, A2AClient
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import (AgentCapabilities, AgentCard, AgentSkill, MessageSendParams, SendMessageRequest, Task,
                       TaskState, UnsupportedOperationError)
from a2a.utils import new_agent_text_message, new_task
from a2a.utils.errors import ServerError
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from openai import OpenAI
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv()
nest_asyncio.apply()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_experientiallabs_llm  # noqa: E402
from lab_kit import load_handbook, load_jsonl, tfidf_retriever  # noqa: E402

DATA = Path("../data/larkspur")
golden = {g["id"]: g for g in load_jsonl(DATA / "golden.jsonl")}
QUESTIONS = ["g05", "g19", "g27"]  # a lookup, a multi-hop and one the handbook cannot answer
llm = get_experientiallabs_llm()
MODEL = llm.model_name
HOST, PORT = "localhost", 10141
BASE_URL = f"http://{HOST}:{PORT}"
print("model:", MODEL, "| server:", BASE_URL)

## The agentic RAG graph (server side)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [c for page in load_handbook(DATA / "handbook") for c in splitter.split_text(page["text"])]
search = tfidf_retriever(chunks, k=4)
MAX_REWRITES = 1


class Grade(BaseModel):
    relevant: Literal["yes", "no"] = Field(description="yes if the passages contain the answer to the question")


class RagState(TypedDict):
    question: str
    query: str
    passages: list[str]
    relevant: str
    rewrites: int
    answer: str


def retrieve(state: RagState) -> dict:
    return {"passages": search(state["query"])}


def grade(state: RagState) -> dict:
    g = llm.with_structured_output(Grade).invoke([
        SystemMessage("Do these handbook passages answer the question?"),
        HumanMessage(f"Question: {state['question']}\n\nPassages:\n" + "\n---\n".join(state["passages"])),
    ])
    return {"relevant": g.relevant}


def rewrite(state: RagState) -> dict:
    q = llm.invoke([SystemMessage("Rewrite the question as a short keyword search query for a cloud host's "
                                  "customer handbook. Reply with the query only."), HumanMessage(state["question"])])
    return {"query": q.content.strip(), "rewrites": state["rewrites"] + 1}


def generate(state: RagState) -> dict:
    if state["relevant"] != "yes":
        return {"answer": "The Larkspur handbook does not answer this question."}
    a = llm.invoke([SystemMessage("Answer from the passages only, in two sentences at most.\n\n"
                                  + "\n---\n".join(state["passages"])), HumanMessage(state["question"])])
    return {"answer": a.content}


def after_grade(state: RagState) -> str:
    return "rewrite" if state["relevant"] != "yes" and state["rewrites"] < MAX_REWRITES else "generate"


g = StateGraph(RagState)
for name, fn in (("retrieve", retrieve), ("grade", grade), ("rewrite", rewrite), ("generate", generate)):
    g.add_node(name, fn)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", after_grade, ["rewrite", "generate"])
g.add_edge("rewrite", "retrieve")
g.add_edge("generate", END)
rag_graph = g.compile()


def ask_graph(question: str) -> str:
    out = rag_graph.invoke({"question": question, "query": question, "passages": [], "relevant": "",
                            "rewrites": 0, "answer": ""})
    return out["answer"]


print(ask_graph(golden["g05"]["question"]))  # local check before wrapping it in A2A

## Serve the graph over A2A

In [ ]:
class AgenticRagExecutor(AgentExecutor):
    """Adapts the compiled agentic RAG graph to the A2A AgentExecutor interface."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task or new_task(context.message)
        if not context.current_task:
            await event_queue.enqueue_event(task)
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.update_status(TaskState.working,
                                    new_agent_text_message("Searching the handbook...", task.context_id, task.id))
        try:
            answer = await asyncio.to_thread(ask_graph, context.get_user_input())
            await updater.update_status(TaskState.completed, new_agent_text_message(answer, task.context_id, task.id))
        except Exception as exc:  # noqa: BLE001
            await updater.update_status(TaskState.failed,
                                        new_agent_text_message(f"agentic RAG error: {exc}", task.context_id, task.id))

    async def cancel(self, request: RequestContext, event_queue: EventQueue) -> Task | None:
        raise ServerError(error=UnsupportedOperationError())


agent_card = AgentCard(
    name="larkspur_handbook_agent",
    description="An agentic RAG agent over the Larkspur Cloud customer handbook. It retrieves, grades and, when "
                "needed, rewrites the query once before answering. It says so when the handbook has no answer.",
    url=f"{BASE_URL}/",
    version="1.0.0",
    defaultInputModes=["text"],
    defaultOutputModes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[AgentSkill(id="handbook_qa", name="Handbook questions",
                       description="Answers Larkspur Cloud questions on billing, refunds, plans, backups, security, "
                                   "API limits, regions, SLA credits and onboarding.",
                       tags=["rag", "support"], examples=["How long does a refund take?"])],
)
app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=DefaultRequestHandler(agent_executor=AgenticRagExecutor(), task_store=InMemoryTaskStore()),
).build()
uvicorn_server = uvicorn.Server(uvicorn.Config(app, host=HOST, port=PORT, log_level="warning"))
server_thread = threading.Thread(target=lambda: asyncio.run(uvicorn_server.serve()), daemon=True)
server_thread.start()
time.sleep(1.5)
print("A2A server for", agent_card.name, "at", BASE_URL)

## The client agent (OpenAI SDK, no framework)

Step one is discovery: the client reads the Agent Card from `/.well-known/agent-card.json` (the resolver's default path in a2a-sdk 0.3; older servers used `/.well-known/agent.json`) and builds its tool description from the card, so it knows what the remote agent does without sharing any code with it.

In [ ]:
async def fetch_card() -> AgentCard:
    async with httpx.AsyncClient(timeout=30.0) as http:
        return await A2ACardResolver(httpx_client=http, base_url=BASE_URL).get_agent_card()


card = asyncio.run(fetch_card())
print(card.name, "|", [s.name for s in card.skills])


def reply_text(payload) -> str:
    """The last text part in an A2A response, whether the result is a Task or a Message."""
    texts = []

    def walk(x):
        if isinstance(x, dict):
            if x.get("kind") == "text" and "text" in x:
                texts.append(x["text"])
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)

    walk(payload)
    return texts[-1] if texts else json.dumps(payload)[:500]


async def send_a2a(text: str) -> str:
    async with httpx.AsyncClient(timeout=300.0) as http:
        client = A2AClient(httpx_client=http, agent_card=card)
        request = SendMessageRequest(id=str(uuid4()), params=MessageSendParams(message={
            "role": "user", "parts": [{"kind": "text", "text": text}], "message_id": str(uuid4())}))
        response = await client.send_message(request)
        return reply_text(response.model_dump(mode="json", exclude_none=True))

In [ ]:
openai_client = OpenAI(api_key=os.environ["EXPERIENTIALLABS_API_KEY"], base_url="https://api.experientiallabs.ai/v1")
TOOLS = [{"type": "function", "function": {
    "name": "ask_handbook_agent",
    "description": f"Ask the remote agent '{card.name}': {card.description}",
    "parameters": {"type": "object", "properties": {"question": {"type": "string"}}, "required": ["question"]},
}}]
MAX_TOOL_ROUNDS = 3


def client_agent(question: str) -> dict:
    messages = [{"role": "system", "content": "You are a customer-success assistant. For anything about Larkspur "
                 "Cloud, ask the handbook agent and pass on its answer faithfully."},
                {"role": "user", "content": question}]
    a2a_calls = []
    for _ in range(MAX_TOOL_ROUNDS):
        msg = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return {"answer": msg.content, "a2a_calls": a2a_calls}
        for call in msg.tool_calls:
            asked = json.loads(call.function.arguments)["question"]
            remote = asyncio.run(send_a2a(asked))
            a2a_calls.append({"asked": asked, "remote_answer": remote})
            messages.append({"role": "tool", "tool_call_id": call.id, "content": remote})
    return {"answer": messages[-1].get("content", ""), "a2a_calls": a2a_calls}

## Three questions through the client agent

In [ ]:
results = []
for gid in QUESTIONS:
    start = time.perf_counter()
    r = client_agent(golden[gid]["question"])
    results.append({"id": gid, "type": golden[gid]["type"], "seconds": round(time.perf_counter() - start, 2), **r})
    print(f"{gid}: {golden[gid]['question']}")
    for c in r["a2a_calls"]:
        print(f"  A2A -> {c['asked']}\n  A2A <- {c['remote_answer']}")
    print(f"  client answer: {r['answer']}\n  reference: {golden[gid]['reference_answer']}\n")

## Results

In [ ]:
import pandas as pd

print("model:", MODEL, "| run:", time.strftime("%Y-%m-%d"))
print(pd.DataFrame([{"id": r["id"], "type": r["type"], "a2a_calls": len(r["a2a_calls"]), "seconds": r["seconds"]}
                    for r in results]).to_markdown(index=False))

In [ ]:
uvicorn_server.should_exit = True
server_thread.join(timeout=5)
print("A2A server stopped.")

## What this shows

- The client knows the remote agent only through its Agent Card: name, description and skills. Swapping the LangGraph graph for any other implementation behind the same card changes nothing on the client side.
- The server returns a task whose status message holds the answer; `reply_text` reads the last text part, so it works whether the server replies with a task or a plain message.
- `g27` checks the refusal path end to end: the graph's grade step says the passages do not answer, and the client agent should pass that on rather than answer from its own knowledge.